# D331 — HBase Practical Lab on Amazon EMR

Use the HBase shell on an existing Amazon EMR `emr-6.15.0` cluster. HBase must be selected as an installed application. Cluster creation is outside this notebook.

**Run location:** commands marked **LOCAL TERMINAL** run on the local computer. Commands at the `hbase(main):...>` prompt run inside the HBase shell on the EMR primary node. Notebook `%%bash` cells work only when the notebook kernel itself runs on that EMR cluster.

**Cost and safety:** an EMR cluster continues to incur charges until terminated. The cleanup at the end removes only the D331 HBase namespace and table; it does not terminate the cluster.

## 1. Connect to the EMR primary node

Find **Primary node public DNS** in the EMR cluster Summary. The primary security group must allow inbound TCP 22 from the connecting IP. Run from the **LOCAL TERMINAL**:

```bash
chmod 400 /path/to/key.pem
ssh -i /path/to/key.pem hadoop@PRIMARY_PUBLIC_DNS
```

The username is `hadoop`; `-i` is followed by the private-key path, not the DNS name. Example shape:

```bash
ssh -i ~/keys/dataeng.pem hadoop@ec2-12-34-56-78.ap-south-1.compute.amazonaws.com
```

Windows PowerShell with built-in OpenSSH uses the same `ssh -i` form. Restrict the `.pem` file permissions if OpenSSH reports that they are too open. Do not commit the private key to a repository.

## 2. Verify release, components, and services

Run these on the **EMR PRIMARY NODE shell** (before entering HBase shell):

```bash
cat /emr/instance-controller/lib/info/extraInstanceData.json | grep releaseLabel
hbase version
sudo systemctl status hbase-master --no-pager
sudo systemctl status zookeeper-server --no-pager
hdfs dfsadmin -report | head -40
```

Expected release is `emr-6.15.0`; AWS packages HBase as `2.4.17-amzn-3`. RegionServers normally run on core nodes, so inspect them with the HMaster UI or HBase shell rather than expecting a local RegionServer on the primary node. Service unit names can vary with packaging; if a unit is not found, use `systemctl list-units | grep -Ei 'hbase|zookeeper'`.

Enter the shell:

```bash
hbase shell
```

Inside it, start with:

```ruby
status 'detailed'
version
whoami
help
```

## 3. Create a namespace and table

A namespace groups tables. A table declares column families up front, while qualifiers remain flexible.

Run in **HBase shell**:

```ruby
create_namespace 'd33'
create 'd33:customer_activity',
  {NAME => 'profile', VERSIONS => 2, BLOOMFILTER => 'ROW'},
  {NAME => 'metrics', VERSIONS => 3}

list_namespace_tables 'd33'
describe 'd33:customer_activity'
```

`VERSIONS` limits how many cell versions are retained after compaction. `BLOOMFILTER => 'ROW'` can help point lookups avoid HFiles that definitely lack the row. Keep column-family counts small because each family has its own MemStore/HFiles and flush/compaction work.

## 4. Put data — one cell at a time

```ruby
put 'd33:customer_activity', 'CUST#1001', 'profile:name', 'Asha'
put 'd33:customer_activity', 'CUST#1001', 'profile:city', 'Chennai'
put 'd33:customer_activity', 'CUST#1001', 'metrics:tier', 'gold'
put 'd33:customer_activity', 'CUST#1002', 'profile:name', 'Ravi'
put 'd33:customer_activity', 'CUST#1002', 'profile:city', 'Bengaluru'
put 'd33:customer_activity', 'CUST#1002', 'metrics:tier', 'silver'
put 'd33:customer_activity', 'CUST#1003', 'profile:name', 'Meera'
put 'd33:customer_activity', 'CUST#1003', 'metrics:tier', 'gold'
```

HBase stores byte arrays. Shell displays convenient strings here, but an application must use consistent serialization for numbers and structured values. Notice that `CUST#1003` has no city: sparse columns need no placeholder.

## 5. Point reads and projections

```ruby
get 'd33:customer_activity', 'CUST#1001'
get 'd33:customer_activity', 'CUST#1001', {COLUMN => 'profile:name'}
get 'd33:customer_activity', 'CUST#1001', {COLUMN => ['profile:name', 'metrics:tier']}
exists 'd33:customer_activity', 'CUST#9999'
```

A point `get` is HBase's natural strength. Supplying only only the required families and qualifiers reduces transferred data. In application clients, reuse connections and batch operations rather than opening a connection per row.

## 6. Ordered scans: always bound them when possible

```ruby
scan 'd33:customer_activity', {LIMIT => 10}
scan 'd33:customer_activity', {STARTROW => 'CUST#1001', STOPROW => 'CUST#1003'}
scan 'd33:customer_activity', {COLUMNS => ['profile:name', 'metrics:tier'], LIMIT => 10}
scan 'd33:customer_activity', {REVERSED => true, LIMIT => 2}
```

`STARTROW` is inclusive and `STOPROW` is exclusive. Scans follow lexicographic byte order, so `CUST#10` sorts before `CUST#2` unless IDs are padded. An unbounded production scan can touch every region and HFile; use a row-key range, projections, caching/batching in clients, and a realistic limit.

## 7. Filters: useful, but not an index

```ruby
scan 'd33:customer_activity', {FILTER => "PrefixFilter('CUST#100')"}
scan 'd33:customer_activity', {
  COLUMNS => ['metrics:tier'],
  FILTER => "SingleColumnValueFilter('metrics','tier',=,'binary:gold')"
}
```

A prefix filter aligns with sorted keys, but a value filter may still scan many rows. Filters reduce results and some processing; they do not provide a general secondary index. If lookup by tier is frequent and selective, model another table such as `tier#customer_id -> customer_id`, or evaluate a suitable indexing/SQL layer.

## 8. Versions and timestamps

Write two explicit versions so the result is repeatable:

```ruby
put 'd33:customer_activity', 'CUST#1001', 'profile:status', 'active', 1700000000000
put 'd33:customer_activity', 'CUST#1001', 'profile:status', 'paused', 1700000001000
get 'd33:customer_activity', 'CUST#1001', {COLUMN => 'profile:status', VERSIONS => 2}
get 'd33:customer_activity', 'CUST#1001', {COLUMN => 'profile:status', TIMESTAMP => 1700000000000}
```

The timestamp is epoch milliseconds. By default a read returns the newest visible version. Version retention is configured by column family and enforced through normal storage lifecycle/compaction; it is not a substitute for audit history.

## 9. Atomic counters and row mutations

```ruby
incr 'd33:customer_activity', 'CUST#1001', 'metrics:login_count', 1
incr 'd33:customer_activity', 'CUST#1001', 'metrics:login_count', 4
get 'd33:customer_activity', 'CUST#1001', {COLUMN => 'metrics:login_count'}
```

An increment is atomic for that cell/row. HBase also supports append and conditional check-and-mutate APIs. Multi-cell mutations within one row can be atomic; do not assume a general transaction across multiple row keys.

## 10. Deletes and tombstones

```ruby
delete 'd33:customer_activity', 'CUST#1002', 'profile:city'
get 'd33:customer_activity', 'CUST#1002'
deleteall 'd33:customer_activity', 'CUST#1003'
count 'd33:customer_activity', INTERVAL => 1
```

Deletes usually create tombstones; compaction later removes eligible cells and tombstones. Consequently, delete-heavy workloads can temporarily use space and make reads work harder. `count` scans the table and is a full-scan command—not a cheap metadata count for large tables.

## 11. Schema lifecycle

Tables must be disabled for some schema operations in HBase 2.x shell workflows:

```ruby
disable 'd33:customer_activity'
alter 'd33:customer_activity', {NAME => 'metrics', VERSIONS => 2, COMPRESSION => 'SNAPPY'}
enable 'd33:customer_activity'
is_enabled 'd33:customer_activity'
describe 'd33:customer_activity'
```

Compression saves storage and I/O at a CPU cost. TTL, version limits, Bloom filters, and compression are column-family properties, so group columns with similar access and retention needs. Benchmark changes; do not copy settings blindly.

## 12. Regions, splits, and hotspot thinking

```ruby
locate_region 'd33:customer_activity', 'CUST#1001'
get_splits 'd33:customer_activity'
status 'simple'
```

Regions are contiguous row-key ranges. They split as they grow and may move between RegionServers. Production bulk-ingest tables can be pre-split, but split points must match the actual key distribution. Too few regions limit parallelism; too many tiny regions add coordination, memory, and compaction overhead.

**Hotspot example:** keys `EVENT#000001`, `EVENT#000002`, ... send new writes to one end of the table. A bounded salt (`00#...` to `0f#...`) spreads writes, but a full time-range read must query all 16 buckets.

## 13. Snapshot before risky work

```ruby
snapshot 'd33:customer_activity', 'd331_customer_activity_v1'
list_snapshots
```

A snapshot is a fast, read-only metadata view that initially references existing HFiles. It is useful before a risky schema/data change, but it still belongs to the cluster's storage environment. For disaster recovery or cluster termination, follow an export/backup process to independent durable storage and test restoration.

Optional restore practice would require disabling the table and would overwrite its current state, so it is intentionally not run here:

```ruby
# disable 'd33:customer_activity'
# restore_snapshot 'd331_customer_activity_v1'
# enable 'd33:customer_activity'
```

## 14. EMR monitoring and troubleshooting

Exit HBase shell with `exit`, then run on the primary node:

```bash
sudo ls -lah /var/log/hbase/
sudo tail -n 100 /var/log/hbase/hbase-hbase-master-*.log
hdfs dfsadmin -report
df -h
```

Useful signals include RegionServer availability, request rate, read/write latency, blocked updates, MemStore size, store-file count, compaction queue, GC pauses, region count/skew, disk usage, and HDFS health. Start with `status 'detailed'`, the HMaster UI, and logs.

HBCK2 is present under `/usr/lib/hbase-operator-tools/` on supported EMR releases, but repair commands can change metadata. Diagnose first and use the exact AWS/Apache procedure for the failure; do not experiment with it on production data.

## 15. Safely view the HMaster UI

Do not open port 16010 to the internet. From the **LOCAL TERMINAL**, establish local forwarding:

```bash
ssh -i /path/to/key.pem -N -L 16010:localhost:16010 hadoop@PRIMARY_PUBLIC_DNS
```

While that session remains open, visit [http://localhost:16010](http://localhost:16010). Inspect live/dead RegionServers, regions in transition, table regions, requests, and store-file metrics. Depending on cluster networking and service binding, AWS's documented SOCKS-proxy approach may be required instead of simple local forwarding.

The EMR HBase UI endpoint is port `16010`; access should remain restricted through SSH/network controls.

## 16. Integration choices

- **Hive mapping:** useful for SQL access to an HBase table; row-key predicate pushdown matters. Not ideal for repeated full scans.
- **Spark:** use an HBase connector/client for parallel transformations; align partitions with regions and avoid one client connection per record.
- **Bulk load:** for very large initial loads, generate correctly partitioned HFiles and use the bulk-load tooling rather than millions of shell `put`s.
- **REST/Thrift:** language-neutral gateways, adding service/security/serialization overhead.
- **Phoenix:** SQL and secondary-index capabilities over HBase, with additional modeling and operational complexity.
- **Replication:** asynchronous cross-cluster replication for selected column families/tables; plan conflict, lag, bandwidth, and recovery behavior.

Use direct shell operations here to expose the native model clearly.

## 17. Cleanup

Run cleanup only after completing all operations. Enter `hbase shell` again if necessary.

```ruby
disable 'd33:customer_activity'
drop 'd33:customer_activity'
delete_snapshot 'd331_customer_activity_v1'
drop_namespace 'd33'
list_namespace
```

A table must be disabled before it is dropped. The namespace must be empty before it is dropped. This removes the D331 data and snapshot; recovery should not be assumed. Cluster termination is a separate action in the EMR console and is intentionally outside this notebook.

## 18. Verification checklist

Verify the following outcomes: 

- connect using `ssh -i KEY.pem hadoop@PRIMARY_PUBLIC_DNS`;
- explain table, row key, family, qualifier, cell, and version;
- perform `put`, `get`, bounded `scan`, filter, increment, and delete;
- explain why row-key order can cause a hotspot;
- distinguish flush, compaction, and split;
- locate EMR HBase logs/UI and explain why ordinary HDFS data must be protected before termination; and
- choose Hive for analytics and HBase for key-oriented operational access.

Official references: [EMR HBase](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-hbase.html), [EMR HBase shell](https://docs.aws.amazon.com/emr/latest/ReleaseGuide/emr-hbase-connect.html), [EMR SSH](https://docs.aws.amazon.com/emr/latest/ManagementGuide/emr-connect-master-node-ssh.html), and [HBase Reference Guide](https://hbase.apache.org/book.html).